In [0]:
%run ../utils/adls_auth

In [0]:
%run ../utils/control_table

In [0]:
%run ../utils/dq_helpers

In [0]:
spark.conf.set("spark.databricks.delta.properties.defaults.enableDeletionVectors", "false")

In [0]:
import uuid
from datetime import datetime
from pyspark.sql.functions import col, to_date, current_timestamp, size, lit, array, arrays_overlap
from delta.tables import DeltaTable

In [0]:
BRONZE_PATH = "abfss://bronze@stdatalakenyctaxi.dfs.core.windows.net/trips_raw"
SCHEMA_LOCATION = "abfss://bronze@stdatalakenyctaxi.dfs.core.windows.net/_schemas/trips_autoloader"
CHECKPOINT_PATH = "abfss://silver@stdatalakenyctaxi.dfs.core.windows.net/_checkpoints/trips_silver"
SILVER_PATH = "abfss://silver@stdatalakenyctaxi.dfs.core.windows.net/trips_silver"
QUARANTINE_PATH = "abfss://silver@stdatalakenyctaxi.dfs.core.windows.net/trips_quarantine"

NATURAL_KEY_COLS = ["VendorID", "tpep_pickup_datetime", "tpep_dropoff_datetime", "PULocationID"]
CRITICAL_CHECKS = {
    "null_pickup_datetime", "null_dropoff_datetime", "null_pickup_location",
    "negative_fare", "negative_distance", "pickup_after_dropoff",
}


In [0]:
if not DeltaTable.isDeltaTable(spark, SILVER_PATH):
    print("Silver trips table doesn't exist yet — will be created by the first batch's initial write.")
    silver_table_exists = False
else:
    silver_table_exists = True


In [0]:

def process_batch(batch_df, batch_id):
    """Called once per Auto Loader micro-batch: dedupe, check, quarantine, MERGE."""
    global silver_table_exists
    run_batch_id = f"{batch_id}_{uuid.uuid4()}"

    before_dedupe = batch_df.count()
    if before_dedupe == 0:
        print(f"[batch {batch_id}] empty batch, skipping.")
        return

    deduped_df = batch_df.dropDuplicates(NATURAL_KEY_COLS)
    duplicates_removed = before_dedupe - deduped_df.count()
    log_dq_result(
        spark, run_batch_id, "silver", "trips_silver", "duplicate_trip_key",
        severity="CRITICAL", rows_checked=before_dedupe, rows_failed=duplicates_removed,
        action_taken="Duplicates dropped within micro-batch via dropDuplicates.",
    )

    checks = {
        "null_pickup_datetime": col("tpep_pickup_datetime").isNull(),
        "null_dropoff_datetime": col("tpep_dropoff_datetime").isNull(),
        "null_pickup_location": col("PULocationID").isNull(),
        "negative_fare": col("fare_amount") < 0,
        "negative_distance": col("trip_distance") < 0,
        "pickup_after_dropoff": col("tpep_pickup_datetime") >= col("tpep_dropoff_datetime"),
        "passenger_count_out_of_range": (col("passenger_count") < 0) | (col("passenger_count") > 8),
    }
    flagged_df = flag_row_level_checks(deduped_df, checks)
    critical_array = array(*[lit(c) for c in CRITICAL_CHECKS])
    flagged_df = flagged_df.withColumn("_has_critical_failure", arrays_overlap(col("_dq_failures"), critical_array))

    quarantine_df = flagged_df.filter(col("_has_critical_failure")).withColumn("_quarantined_at", current_timestamp())
    clean_df = flagged_df.filter(~col("_has_critical_failure")).withColumn("pickup_date", to_date(col("tpep_pickup_datetime")))

        # --- Monitor Auto Loader's rescued-data safety net ---
    if "_rescued_data" in clean_df.columns:
        rescued_count = clean_df.filter(col("_rescued_data").isNotNull()).count()
        if rescued_count > 0:
            log_dq_result(
                spark, run_batch_id, "silver", "trips_silver", "rescued_data_present",
                severity="WARNING", rows_checked=clean_df.count(), rows_failed=rescued_count,
                action_taken="Rows contain unrecognized fields in _rescued_data — investigate source schema before next run.",
            )
            print(f"WARNING: {rescued_count} rows have unrecognized fields in _rescued_data.")




    critical_failed = quarantine_df.count()
    log_dq_result(
        spark, run_batch_id, "silver", "trips_silver", "critical_row_checks",
        severity="CRITICAL", rows_checked=before_dedupe, rows_failed=critical_failed,
        action_taken="Rows with any critical failure routed to trips_quarantine, excluded from MERGE.",
    )

    # --- MERGE into Silver (incremental upsert on natural key) ---
    if not silver_table_exists:
        clean_df.write.format("delta").mode("overwrite").partitionBy("pickup_date").save(SILVER_PATH)
        silver_table_exists = True
    else:
        silver_table = DeltaTable.forPath(spark, SILVER_PATH)
        merge_condition = " AND ".join([f"target.{c} = source.{c}" for c in NATURAL_KEY_COLS])

        spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled", "true")
        (silver_table.alias("target")
            .merge(clean_df.alias("source"), merge_condition)
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute())
        
        spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled", "false")  

    quarantine_df.write.format("delta").mode("append").save(QUARANTINE_PATH)

    log_ingestion_event(
        spark, run_batch_id, "trips_silver_transform", f"batch_{batch_id}",
        status="SUCCESS", rows_written=clean_df.count(), started_at=datetime.utcnow(),
    )
    print(f"[batch {batch_id}] merged {clean_df.count()} rows, quarantined {critical_failed}, deduped {duplicates_removed}.")



In [0]:
# --- Auto Loader read: incremental, schema-evolution-aware ---
autoloader_stream = (
    spark.readStream.format("cloudFiles")
    .option("cloudFiles.format", "parquet")
    .option("cloudFiles.schemaLocation", SCHEMA_LOCATION)
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")  
    .option("mergeSchema", "true")
    .load(BRONZE_PATH)
)

query = (
    autoloader_stream.writeStream
    .foreachBatch(process_batch)
    .option("checkpointLocation", CHECKPOINT_PATH)
    .trigger(availableNow=True)  
    .start()
)


try:
    query.awaitTermination()
    print("Trips Silver run complete (Auto Loader caught up, query terminated).")
except Exception as e:
    error_text = str(e)
    if "SCHEMA" in error_text.upper() or "SchemaColumnTypeException" in error_text or "DELTA_FAILED_TO_MERGE_FIELDS" in error_text.upper():
        log_dq_result(
            spark, str(uuid.uuid4()), "silver", "trips_silver", "breaking_schema_change_detected",
            severity="CRITICAL", rows_checked=0, rows_failed=0,
            action_taken=f"Pipeline halted — non-additive schema change detected: {error_text[:500]}",
        )
        print(f"CRITICAL: breaking schema change detected, pipeline halted.\n{error_text}")
    raise